<a href="https://colab.research.google.com/github/meganttp/PLOS-Prediction-Healthcare/blob/main/DAT6000_PLOS_Modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd

# Connect Gogle drive
path = '/content/drive/MyDrive/DATA6000-A1/02Exploratory/Final Cleaning_30-70+Hospital_Inpatient_Discharges_(SPARCS_De-Identified)__2021_20260405).csv'

df = pd.read_csv(path)

print(f"Data Loaded: {df.shape}")
df.head()

Data Loaded: (602176, 15)


,Age Group,Gender,Length of Stay,Type of Admission,Patient Disposition,CCSR Diagnosis Description,CCSR Procedure Description,APR DRG Description,APR Severity of Illness Code,APR Severity of Illness Description,APR Risk of Mortality,APR Medical Surgical Description,Emergency Department Indicator,Total Charges,Total Costs
0,70 or Older,M,27,Emergency,Home w/ Home Health Services,COVID-19,ISOLATION PROCEDURES,MAJOR RESPIRATORY INFECTIONS AND INFLAMMATIONS,3,Major,Extreme,Medical,Y,"$320,922.43","$60,241.34"
1,70 or Older,M,5,Emergency,Home or Self Care,Urinary tract infections,ADMINISTRATION OF NUTRITIONAL AND ELECTROLYTIC...,KIDNEY AND URINARY TRACT INFECTIONS,3,Major,Major,Medical,Y,"$72,700.17","$12,111.75"
2,50 to 69,F,3,Emergency,Home or Self Care,Paralysis (other than cerebral palsy),LUMBAR PUNCTURE,OTHER DISORDERS OF NERVOUS SYSTEM,2,Moderate,Minor,Medical,Y,"$55,562.51","$8,339.72"
3,50 to 69,M,6,Emergency,Home or Self Care,Complication of other surgical or medical care...,COMPUTERIZED TOMOGRAPHY (CT) WITHOUT CONTRAST,OTHER COMPLICATIONS OF TREATMENT,3,Major,Moderate,Medical,Y,"$109,269.27","$18,443"
4,50 to 69,M,21,Emergency,Left Against Medical Advice,Complication of other surgical or medical care...,LARYNGOSCOPY (DIAGNOSTIC),"POST-OPERATIVE, POST-TRAUMATIC, OTHER DEVICE I...",2,Moderate,Moderate,Medical,Y,"$270,656.16","$48,268.17"


In [16]:
print('Original number of rows:', df.shape[0])

# Define the list of admission types to keep
admission_types_to_keep = ['Elective', 'Emergency', 'Trauma', 'Urgent']

# Filter the DataFrame
df_filtered = df[df['Type of Admission'].isin(admission_types_to_keep)].copy()

print('Number of rows after filtering:', df_filtered.shape[0])
print('First 5 rows of the filtered DataFrame:')
display(df_filtered.head())

Original number of rows: 602176
Number of rows after filtering: 601819
First 5 rows of the filtered DataFrame:


,Age Group,Gender,Length of Stay,Type of Admission,Patient Disposition,CCSR Diagnosis Description,CCSR Procedure Description,APR DRG Description,APR Severity of Illness Code,APR Severity of Illness Description,APR Risk of Mortality,APR Medical Surgical Description,Emergency Department Indicator,Total Charges,Total Costs
0,70 or Older,M,27,Emergency,Home w/ Home Health Services,COVID-19,ISOLATION PROCEDURES,MAJOR RESPIRATORY INFECTIONS AND INFLAMMATIONS,3,Major,Extreme,Medical,Y,"$320,922.43","$60,241.34"
1,70 or Older,M,5,Emergency,Home or Self Care,Urinary tract infections,ADMINISTRATION OF NUTRITIONAL AND ELECTROLYTIC...,KIDNEY AND URINARY TRACT INFECTIONS,3,Major,Major,Medical,Y,"$72,700.17","$12,111.75"
2,50 to 69,F,3,Emergency,Home or Self Care,Paralysis (other than cerebral palsy),LUMBAR PUNCTURE,OTHER DISORDERS OF NERVOUS SYSTEM,2,Moderate,Minor,Medical,Y,"$55,562.51","$8,339.72"
3,50 to 69,M,6,Emergency,Home or Self Care,Complication of other surgical or medical care...,COMPUTERIZED TOMOGRAPHY (CT) WITHOUT CONTRAST,OTHER COMPLICATIONS OF TREATMENT,3,Major,Moderate,Medical,Y,"$109,269.27","$18,443"
4,50 to 69,M,21,Emergency,Left Against Medical Advice,Complication of other surgical or medical care...,LARYNGOSCOPY (DIAGNOSTIC),"POST-OPERATIVE, POST-TRAUMATIC, OTHER DEVICE I...",2,Moderate,Moderate,Medical,Y,"$270,656.16","$48,268.17"


## **Chart 1: Heatmap: Probability of PLOS by Admission Type and Severity**

In [26]:
import plotly.express as px
import pandas as pd

# Define the desired order for APR Severity of Illness
severity_order = ['Minor', 'Moderate', 'Major', 'Extreme']

# Convert 'Length of Stay' to numeric, handling potential errors and non-numeric characters
df_filtered['Length of Stay'] = pd.to_numeric(
    df_filtered['Length of Stay'].astype(str).str.replace(r'[^0-9.]', '', regex=True),
    errors='coerce'
)

# Group by 'Type of Admission' and 'APR Severity of Illness Description' and calculate the mean 'Length of Stay'
# Handle missing values in 'Length of Stay' before grouping and use 'df_filtered'
heatmap_data = df_filtered.dropna(subset=['Length of Stay']).groupby(['Type of Admission', 'APR Severity of Illness Description'])['Length of Stay'].mean().reset_index()

# Convert 'APR Severity of Illness Description' to a categorical type with the specified order
heatmap_data['APR Severity of Illness Description'] = pd.Categorical(
    heatmap_data['APR Severity of Illness Description'],
    categories=severity_order,
    ordered=True
)

# Create the heatmap
fig = px.density_heatmap(
    heatmap_data,
    x='APR Severity of Illness Description',
    y='Type of Admission',
    z='Length of Stay',
    title='Average Length of Stay by Type of Admission and APR Severity of Illness',
    labels={
        'APR Severity of Illness Description': 'APR Severity of Illness',
        'Type of Admission': 'Type of Admission',
        'Length of Stay': 'Average Length of Stay'
    },
    color_continuous_scale='RdBu_r', # Changed to RdBu_r for blue=lower LOS, red=higher LOS
    text_auto=".2f" # Show the 'Length of Stay' numbers on the heatmap cells, formatted to 2 decimal places
)

# Adjust layout for better readability and enforce x-axis order
fig.update_layout(
    xaxis_title_standoff=10,
    yaxis_title_standoff=10,
    xaxis=dict(
        tickangle=-45,
        automargin=True,
        categoryorder='array', # Ensure categories are ordered by an array
        categoryarray=severity_order # Specify the order of categories
    ),
    yaxis=dict(automargin=True),
    margin=dict(l=100, r=100, t=100, b=100),
    width=900, # Set width for 1:1 aspect ratio
    height=700 # Set height for 1:1 aspect ratio
)

fig.show()

## **Chart 2: Interactive Scatter Plot: Total Costs vs. Length of Stay by Top 20 APR DRG Descriptions**

In [33]:
import plotly.express as px
import pandas as pd

# Ensure 'Total Costs' is numeric, cleaning non-numeric characters
df_filtered['Total Costs'] = pd.to_numeric(
    df_filtered['Total Costs'].astype(str).str.replace(r'[^0-9.]', '', regex=True),
    errors='coerce'
)

# Drop rows where 'Total Costs' or 'Length of Stay' might be NaN after conversion
df_scatter = df_filtered.dropna(subset=['Total Costs', 'Length of Stay']).copy()

# Identify top 20 'APR DRG Description' groups based on frequency
top_drg_descriptions = df_scatter['APR DRG Description'].value_counts().nlargest(20).index

# Filter the DataFrame to include only these top groups
df_scatter_top_drg = df_scatter[df_scatter['APR DRG Description'].isin(top_drg_descriptions)]

# Aggregate data for the scatter plot
aggregated_scatter_data = df_scatter_top_drg.groupby('APR DRG Description').agg(
    Avg_Total_Costs=('Total Costs', 'mean'),
    Avg_Length_of_Stay=('Length of Stay', 'mean'),
    Count_of_Rows=('APR DRG Description', 'count')
).reset_index()

# Create the interactive scatter plot with aggregated data
fig_scatter = px.scatter(
    aggregated_scatter_data,
    x='Avg_Total_Costs',
    y='Avg_Length_of_Stay',
    size='Count_of_Rows', # Size of bubbles based on the count of rows
    text='APR DRG Description', # Add text labels to bubbles
    color='APR DRG Description', # Color by DRG descriptions
    hover_name='APR DRG Description', # Show DRG description on hover
    hover_data={
        'Avg_Total_Costs': ':.2f',
        'Avg_Length_of_Stay': ':.2f',
        'Count_of_Rows': True
    }, # Format hover data
    title='Average Total Costs vs. Average Length of Stay by Top 20 APR DRG Descriptions',
    labels={
        'Avg_Total_Costs': 'Average Total Costs',
        'Avg_Length_of_Stay': 'Average Length of Stay',
        'APR DRG Description': 'APR DRG Description',
        'Count_of_Rows': 'Number of Patients'
    },
    height=700, # Adjust height for better visibility
    width=1000,  # Adjust width
    log_x=True, # Use logarithmic scale for x-axis due to potentially wide range of costs
    template='plotly_white',
    size_max=60 # Make bubbles larger and easy to read
)

fig_scatter.update_traces(
    textposition='top center', # Position text labels
    marker=dict(line=dict(width=1, color='DarkSlateGrey')) # Add border to bubbles for better visibility
)
fig_scatter.update_layout(
    legend_title_text='APR DRG Description',
    xaxis_title_standoff=10,
    yaxis_title_standoff=10
)

fig_scatter.show()

## **Chart 3: Interactive Scatter Plot: High-Volume Operational Drivers**

In [36]:
import plotly.express as px

# Filter the aggregated_scatter_data for high-volume DRGs (Count_of_Rows > 5500)
high_volume_drgs_filtered = aggregated_scatter_data[aggregated_scatter_data['Count_of_Rows'] > 5500].copy()

# Create the interactive scatter plot with filtered high-volume data
fig_high_volume_scatter = px.scatter(
    high_volume_drgs_filtered,
    x='Avg_Total_Costs',
    y='Avg_Length_of_Stay',
    size='Count_of_Rows', # Size of bubbles based on the count of rows
    text='APR DRG Description', # Add text labels to bubbles
    color='APR DRG Description', # Color by DRG descriptions
    hover_name='APR DRG Description', # Show DRG description on hover
    hover_data={
        'Avg_Total_Costs': ':.2f',
        'Avg_Length_of_Stay': ':.2f',
        'Count_of_Rows': True
    }, # Format hover data
    title='Average Total Costs vs. Average Length of Stay for High-Volume DRGs (Count > 5500)',
    labels={
        'Avg_Total_Costs': 'Average Total Costs',
        'Avg_Length_of_Stay': 'Average Length of Stay',
        'APR DRG Description': 'APR DRG Description',
        'Count_of_Rows': 'Number of Patients'
    },
    height=700, # Adjust height for better visibility
    width=1000,  # Adjust width
    log_x=True, # Use logarithmic scale for x-axis due to potentially wide range of costs
    template='plotly_white',
    size_max=60 # Make bubbles larger and easy to read
)

fig_high_volume_scatter.update_traces(
    textposition='top center', # Position text labels
    marker=dict(line=dict(width=1, color='DarkSlateGrey')) # Add border to bubbles for better visibility
)
fig_high_volume_scatter.update_layout(
    legend_title_text='APR DRG Description',
    xaxis_title_standoff=10,
    yaxis_title_standoff=10
)

fig_high_volume_scatter.show()


In [56]:
import plotly.express as px
import pandas as pd # Ensure pandas is imported

# Clean 'Length of Stay' and 'Total Costs' columns in df
df['Length of Stay'] = pd.to_numeric(
    df['Length of Stay'].astype(str).str.replace(r'[^0-9.]', '', regex=True),
    errors='coerce'
)
df['Total Costs'] = pd.to_numeric(
    df['Total Costs'].astype(str).str.replace(r'[^0-9.]', '', regex=True),
    errors='coerce'
)

# Drop rows with NaN in 'Length of Stay' or 'Total Costs' after conversion
df_cleaned = df.dropna(subset=['Length of Stay', 'Total Costs']).copy()

# 1. Aggregate (APR DRG Description)
df_grouped = df_cleaned.groupby('APR DRG Description').agg(
    Avg_Length_of_Stay=('Length of Stay', 'mean'),
    Avg_Total_Cost=('Total Costs', 'mean'),
    Patient_Count=('APR DRG Description', 'count')
).reset_index()

# 2. Filter Row > 5,500
df_top_diseases = df_grouped[df_grouped['Patient_Count'] > 5500]

# 3. Create Scatter Plot
fig = px.scatter(
    df_top_diseases,
    x='Avg_Total_Cost',
    y='Avg_Length_of_Stay',
    size='Patient_Count',      # Size depends on diagnoses frequency
    color='Patient_Count', # Colour ranged by Patient_Count
    color_continuous_scale='RdBu_r', # Changed to RdBu_r for blue to red gradient
    hover_name='APR DRG Description',
    text='APR DRG Description', # Put APR Description
    title='High Volume Diseases (>5,000 patients) by Cost and Stay',
    labels={
        'Avg_Total_Cost': 'Average Total Cost ($)',
        'Avg_Length_of_Stay': 'Average Length of Stay (Days)',
        'Patient_Count': 'Number of Patients'
    },
    template='plotly_white',
    size_max=45
)

# Adjust text
fig.update_traces(textposition='top center')
fig.update_layout(height=800)

fig.show()